In [ ]:
#| default_exp app

# App

> FastHTML webhook routes, deduplication, logging setup, and uvicorn server startup.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import asyncio, os, threading, time
import uvicorn
from fasthtml.common import *
from starlette.requests import Request
from starlette.responses import PlainTextResponse
import xml.etree.ElementTree as ET

from solveit_wxbot.config import TOKEN, DEDUP_WINDOW
from solveit_wxbot.crypto import msg_sig, decrypt
from solveit_wxbot.dialog import process_message

## 消息去重

WeCom 在未收到及时响应时会重发消息。`_is_dup` 用内存字典记录最近 `DEDUP_WINDOW` 秒内已处理的 `MsgId`，同时清理过期条目，防止重复触发 AI 调用。

In [ ]:
#| export
_seen: dict[str, float] = {}

def _is_dup(
    mid: str  # WeCom MsgId to check
) -> bool:
    "Return True if `mid` was seen within the last `DEDUP_WINDOW` s (deduplicates WeCom retries)."
    now = time.time()
    for k in [k for k, v in _seen.items() if now - v > DEDUP_WINDOW]: del _seen[k]
    if mid in _seen: return True
    _seen[mid] = now
    return False

## XML 解析

WeCom 消息体为 XML 格式，`_parse_xml` 将其转为普通字典，简化后续字段提取。

In [ ]:
#| export
def _parse_xml(
    s: str  # XML string from WeCom
) -> dict:
    "Parse a WeCom XML message into a `{tag: text}` dict."
    return {c.tag: c.text for c in ET.fromstring(s)}

## Webhook 路由

- **GET `/wecom`**：WeCom 服务器 URL 验证，校验签名后返回解密的 `echostr`。
- **POST `/wecom`**：接收加密消息，验签、解密、去重后，将文本消息异步交给 `process_message` 处理，立即返回空响应以满足 WeCom 3秒超时要求。

In [ ]:
#| export
_app, _rt = fast_app(port=8000)

@_rt('/wecom', methods=['GET'])
async def _wecom_verify(request: Request):
    p = request.query_params
    sig, ts, nonce, echo = p.get('msg_signature',''), p.get('timestamp',''), p.get('nonce',''), p.get('echostr','')
    if msg_sig(TOKEN, ts, nonce, echo) != sig:
        return PlainTextResponse('signature mismatch', status_code=403)
    decrypted, _ = decrypt(echo)
    return PlainTextResponse(decrypted)

@_rt('/wecom', methods=['POST'])
async def _wecom_receive(request: Request):
    p = request.query_params
    sig, ts, nonce = p.get('msg_signature',''), p.get('timestamp',''), p.get('nonce','')
    body = await request.body()
    try:
        outer = _parse_xml(body.decode('utf-8'))
        enc = outer.get('Encrypt', '')
        if msg_sig(TOKEN, ts, nonce, enc) != sig:
            print(f'❌ 签名验证失败')
            return PlainTextResponse('signature mismatch', status_code=403)
        xml_text, corp_id = decrypt(enc)
        msg = _parse_xml(xml_text)
        msg_type, mid = msg.get('MsgType',''), msg.get('MsgId','')
        user_id, content = msg.get('FromUserName',''), msg.get('Content','')
        if msg_type == 'text' and content and not _is_dup(mid):
            asyncio.create_task(process_message(user_id, content))
    except Exception as e:
        print(f'❌ 解析失败: {e}')
    return PlainTextResponse('')

## 服务启动与停止

`start_bot` 在后台守护线程中运行 uvicorn，返回 server 实例供后续调用 `stop_bot` 优雅关闭，便于在 notebook 中交互式控制服务生命周期。

In [ ]:
#| export
def start_bot() -> uvicorn.Server:
    "Start the uvicorn server in a daemon thread and return the server instance."
    config = uvicorn.Config(
        _app,
        host='0.0.0.0',
        port=8000,
        reload=True,
        access_log=False,
    )
    server = uvicorn.Server(config)
    threading.Thread(target=server.run, daemon=True).start()
    return server

In [ ]:
#| export
def stop_bot(server: uvicorn.Server) -> None:
    "Signal the uvicorn server to shut down gracefully."
    server.should_exit = True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()